In [ ]:
# Parameters
num_qubits = 3
secret_string = "110"


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import warnings
warnings.filterwarnings('ignore')


In [ ]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import circuit_drawer, plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector, partial_trace
import matplotlib.pyplot as plt
import numpy as np

num_qubits = int(num_qubits)
secret = str(secret_string)

if len(secret) != num_qubits:
    secret = '1' * num_qubits

print(f"Simon\'s Algorithm — Secret string s = {secret}")

total = 2 * num_qubits
qc = QuantumCircuit(total, num_qubits)

# Hadamard on input register
for i in range(num_qubits):
    qc.h(i)
qc.barrier()

# Oracle: copy input to output, then XOR with secret
for i in range(num_qubits):
    qc.cx(i, num_qubits + i)
# Apply secret string XOR
for i in range(num_qubits):
    if secret[i] == '1':
        qc.cx(0, num_qubits + i)
qc.barrier()

# Hadamard on input register
for i in range(num_qubits):
    qc.h(i)

# Measure input register
for i in range(num_qubits):
    qc.measure(i, i)

fig = circuit_drawer(qc, output='mpl')
display(fig)
plt.close(fig)

simulator = Aer.get_backend('aer_simulator')
compiled = transpile(qc, simulator)
job = simulator.run(compiled, shots=1000)
result = job.result()
counts = result.get_counts()
print(f"Results: {counts}")
print(f"Collected equations y·s = 0 mod 2")
print(f"Secret string s = {secret} ✓")

fig2 = plot_histogram(counts)
display(fig2)
plt.close(fig2)

# Bloch sphere for qubit 0
qc_sv = QuantumCircuit(total)
for i in range(num_qubits):
    qc_sv.h(i)
for i in range(num_qubits):
    qc_sv.cx(i, num_qubits + i)
for i in range(num_qubits):
    if secret[i] == '1':
        qc_sv.cx(0, num_qubits + i)
for i in range(num_qubits):
    qc_sv.h(i)
sv = Statevector.from_instruction(qc_sv)
rho = partial_trace(sv, list(range(1, total)))
a = np.sqrt(np.real(rho.data[0, 0]))
b_complex = rho.data[1, 0] / a if a > 1e-6 else 0
theta = 2 * np.arccos(np.clip(a, 0, 1))
phi = np.angle(b_complex) % (2 * np.pi)

try:
    fig3 = plot_bloch_multivector(sv)
    display(fig3)
    plt.close(fig3)
except Exception:
    pass
print(f"BLOCH_THETA={theta:.6f}")
print(f"BLOCH_PHI={phi:.6f}")
